In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(rcartocolor)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
library(clustree)
getwd()

dataset_id <- "10xHuman_8CLC"
sample_id <- "DUX4"
dir.create("figures_10xHuman_8CLC")
dir.create("data_10xHuman_8CLC")

In [ ]:
# colorDict = c("Differentiating"="#88535A",
#               "2CLC"="#EF8264",
#               "Pluripotent"="#F2CC8F")

# getwd()


# Pre

In [ ]:

conversionTable <- read.table("annotation/annotation_hg38_conversion_withAge.tsv") # created with annotation_scripts/create_annotations_human.Rmd
head(conversionTable)

In [ ]:
SoloTE_path <- paste0("/mnt/volume_1p5T/results/SoloTEout/", dataset_id, "/", sample_id, "/", sample_id,"_SoloTE_output/", sample_id,"_legacytes_MATRIX")

# get filtered barcodes
STAR_path <- paste0("/mnt/TEresults/snakemake_results/results/STARoutdir/",dataset_id,"/",sample_id,"/best_Solo.out/Gene")
filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo

# 2CLC sample
legacyTEmatrix <- Seurat::ReadMtx(mtx = paste0(SoloTE_path, "/matrix.mtx"), 
                              cells = paste0(SoloTE_path, "/barcodes.tsv"), 
                              features = paste0(SoloTE_path, "/features.tsv")) # read matrix

legacyTEmatrix <- legacyTEmatrix[,filteredBarcodes]

# select TEs 
TEs <- grep("SoloTE", rownames(legacyTEmatrix), value = T)
locusTEs <- grep("chr", TEs, value = T)
# subset the matrix keeping only TEs
TEmatrix <- legacyTEmatrix[locusTEs,]
# remove "SoloTE" from the name of the TEs

rownames(TEmatrix) <- gsub("SoloTE\\|", "", rownames(TEmatrix))

# rownames(TEmatrix) <- gsub("\\|", "-", rownames(TEmatrix))
# rownames(TEmatrix) <- gsub("\\_", "-", rownames(TEmatrix))
# rownames(TEmatrix) <- gsub("\\?", "", rownames(TEmatrix))

# table(rownames(TEmatrix) %in% conversionTable$soloteID)
# # transform into Stellarscope IDs
# rownames(TEmatrix) <- conversionTable$stellarscopeID[match(rownames(TEmatrix), conversionTable$soloteID)]

nCells <- ncol(TEmatrix)
thrMinCells <- round(nCells * 0.02)
# create Seurat object with shallow filtering of TEs expressed in at least 50 cells and cells expressing at least 50 TEs 
objTE <- Seurat::CreateSeuratObject(TEmatrix, project = "SoloTE", 
            min.cells = thrMinCells, min.features = 0) 
objTE
thrMinCells

In [ ]:
objTE_SoloTE <- objTE#[,filteredBarcodes] # keep only filtered barcodes
objTE_SoloTE

In [ ]:
# select genes
genes <- setdiff(rownames(legacyTEmatrix), TEs)
# subset the matrix keeping only genes
Genematrix <- legacyTEmatrix[genes,]
# create Seurat object with shallow filtering of genes expressed in at least 50 cells and cells expressing at least 100 genes 
Gene <- Seurat::CreateSeuratObject(Genematrix, project = "8CLC", 
                          min.cells = thrMinCells, min.features = 100) 


objGenes_SoloTE <- Gene #[,filteredBarcodes] # keep only filtered barcodes
objGenes_SoloTE

In [ ]:
# QC TEs

options(repr.plot.width=7, repr.plot.height=6)

objTE_SoloTE@meta.data$nCount_TE <- objTE_SoloTE@meta.data$nCount_RNA 
objTE_SoloTE@meta.data$nFeature_TE <- objTE_SoloTE@meta.data$nFeature_RNA 

# Visualize QC metrics as a violin plot
VlnPlot(objTE_SoloTE, features = c("nCount_RNA", "nFeature_RNA"), ncol = 2, 
        pt.size = 0, alpha = 0.5) + geom_hline(yintercept = 400)
summary(objTE_SoloTE$nCount_RNA)
summary(objTE_SoloTE$nFeature_RNA)

In [ ]:
# QC genes
options(repr.plot.width=8, repr.plot.height=5)

VlnPlot(objGenes_SoloTE, features = c( "nCount_RNA", "nFeature_RNA" ), ncol = 2, 
         pt.size = 0.05, alpha = 0.5) + 
  geom_hline(yintercept = 2000)

summary(objGenes_SoloTE$nFeature_RNA)
summary(objGenes_SoloTE$nCount_RNA)

# Genes

In [ ]:
gc()

objGenes_SoloTE <- NormalizeData(objGenes_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

objGenes_SoloTE <- FindVariableFeatures(objGenes_SoloTE, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objGenes_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objGenes_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:
gc()
all.genes <- rownames(objGenes_SoloTE)
objGenes_SoloTE <- ScaleData(objGenes_SoloTE) # on hvgs

objGenes_SoloTE <- RunPCA(objGenes_SoloTE, features = VariableFeatures(object = objGenes_SoloTE))


DimPlot(objGenes_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objGenes_SoloTE)


In [ ]:
objGenes_SoloTE <- FindNeighbors(objGenes_SoloTE, dims = 1:10, k.param = 20)
objGenes_SoloTE <- FindClusters(objGenes_SoloTE, resolution = 1, algorithm = 4)
objGenes_SoloTE <- RunUMAP(objGenes_SoloTE, dims = 1:10)
DimPlot(objGenes_SoloTE, reduction = "umap")

In [ ]:

options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objGenes_SoloTE, reduction = "umap", group.by="RNA_snn_res.1",
        #cols=colorDict, 
        shuffle=T, pt.size = 0.5) + 
        theme_pubr() +
  theme(text=element_text(size=20)) 
ggsave(paste0("figures_",dataset_id,"/umap_genes_clusters.pdf"), device = "pdf", width=6, height=5)

# TEs 

In [ ]:

### Normalize

objTE_SoloTE <- NormalizeData(objTE_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

#saveRDS(objTE_SoloTE, file=paste0("data_",dataset_id,"/objTE_SoloTE_beforeHVG.RDS"))



In [ ]:

objTE_SoloTE <- FindVariableFeatures(objTE_SoloTE, selection.method = "vst", nfeatures = 4000)

# # Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_SoloTE)
objTE_SoloTE <- ScaleData(objTE_SoloTE) # on hvgs

grep("ERVL", all.genes, value = T)[1:50]


In [ ]:

objTE_SoloTE <- RunPCA(objTE_SoloTE, features = VariableFeatures(object = objTE_SoloTE))


DimPlot(objTE_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objTE_SoloTE)

In [ ]:
objTE_SoloTE <- FindNeighbors(objTE_SoloTE, dims = 1:13, k.param = 20)
objTE_SoloTE <- FindClusters(objTE_SoloTE, resolution = 1, algorithm=4)
objTE_SoloTE <- RunUMAP(objTE_SoloTE, dims = 1:13)
DimPlot(objTE_SoloTE, reduction = "umap")

In [ ]:
options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "seurat_clusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  scale_color_carto_d(palette = "Pastel") +
  theme(text=element_text(size=20)) 
ggsave(paste0("figures_", dataset_id, "/umap_locus_clusters.pdf"), device = "pdf", width=6, height=5)


In [ ]:
# # Save objects
saveRDS(objGenes_SoloTE, paste0("data_", dataset_id, "/soloTE_", dataset_id, "_GENE_seuratObj.RDS"))
saveRDS(objTE_SoloTE, paste0("data_", dataset_id, "/soloTE_", dataset_id, "_seuratObj.RDS"))